## GazeStateNet Demo: 3D Gaze Estimation with Gaze-State Features

This notebook demonstrates gaze direction inference using **GazeStateNet**, which enhances the original GAFA framework with five gaze-state temporal features: Gf (fixation frequency), Gd (gaze density), Ga (head stability), Gv (head-body correlation), and Gs (spatial entropy).

**Key features:**
1. Frozen pretrained HBNet → prevents overfitting, 770K trainable params
2. 5 RHFD features computed from head_dir + body_dv (no extra labels)
3. Original GAFA-compatible LSTM architecture
4. Stable training validated on full GAFA benchmark

In [ ]:
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt

from models.gazenet import GazeStateNet
from dataloader.gafa import create_gafa_dataset

### Load pretrained model

In [ ]:
### Load GazeStateNet

model = GazeStateNet(n_frames=7)

# Load pretrained GAFA weights (HBNet only, GazeModule randomly initialized)
model.load_pretrained_hbnet('./models/weights/gazenet_GAFA.pth')

model.cuda()
model.eval()

print(f"Total parameters:    {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"HBNet frozen:        {not any(p.requires_grad for p in model.hbnet.parameters())}")
print(f"RHFD features:       Gf, Gd, Ga, Gv, Gs")
print(f"LSTM input dim:      11 (body_dir + head_dir + 5 RHFD)")

### Load dataset


Our model takes body image, head position, body velocity as inputs. The following form of input is assumedWe assume following input. Note that our model revieves multiple frames and estimates gaze direction for each frame. 
   - body image: Image capturing whole body of a person. torch.Tensor with the size of (number of frames x 3 (RGB channel) x 256 (image height) x 192 (image width))
   - head mask: Binary image indicating the head position of the person. torch.Tensor with the size of (number of frames x 1 x 256 x 192)
   - body velocity: 2D body velocity of body movement in image plane. torch.Tensor with the size of (number of frames x 2 (dx, dy))

Please see the preprocessing script for details [dataloader/gafa.py](dataloader/gafa.py).

In [58]:
sequence = ['living_room/006']

dataset = create_gafa_dataset(n_frames=7, exp_names=sequence, root_dir='./data/preprocessed/')

In [59]:
batch = dataset[0]

print(batch['image'].shape)
print(batch['head_mask'].shape)
print(batch['body_dv'].shape)

torch.Size([7, 3, 256, 192])
torch.Size([7, 1, 256, 192])
torch.Size([7, 2])


### Inference

In [ ]:
image, head_mask, body_dv = batch['image'], batch['head_mask'], batch['body_dv']

with torch.no_grad():
    image = image.cuda().unsqueeze(0)
    head_mask = head_mask.cuda().unsqueeze(0)
    body_dv = body_dv.cuda().unsqueeze(0)
    gaze_res, head_res, body_res = model(image, head_mask, body_dv)

gaze_direction = gaze_res['direction'][0].cpu()          # [T, 3] predicted direction
gaze_confidence = gaze_res['kappa'][0].cpu()              # [T, 1] confidence

print("Gaze direction shape:", gaze_direction.shape)
print("Gaze confidence shape:", gaze_confidence.shape)
print("Center frame direction:", gaze_direction[3].numpy())
print("Center frame confidence:", gaze_confidence[3].item())

### Visualization

Plot the estimated gaze direction projected onto the image plane.

In [ ]:
# Project 3D gaze to 2D image plane
gaze_dir_2d = gaze_direction[3, 0:2].numpy()  # center frame
gaze_dir_2d /= np.linalg.norm(gaze_dir_2d)

In [62]:
def denormalize(image):
    return image.transpose(1, 2, 0) * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])

In [ ]:
vis_image = denormalize(batch['image'].numpy()[0])
head_center = (70, 40)  # approximate head center

des = (head_center[0] + int(gaze_dir_2d[0]*50), head_center[1] + int(gaze_dir_2d[1]*50))

vis_image = cv2.arrowedLine(vis_image.copy(), head_center, des, (0, 255, 0), 3, tipLength=0.3)

plt.figure(figsize=(6, 6))
plt.imshow(vis_image)
plt.title("GazeStateNet Prediction")
plt.axis('off')
plt.show()